In [1]:
import sys

sys.path.append("../src")

from pathlib import Path

import pandas as pd
import truststore
from entsoe.entsoe import EntsoePandasClient
from entsoe.exceptions import NoMatchingDataError

from energy_load_forecast.config import get_entsoe_api_key

In [2]:
api_key = get_entsoe_api_key()
country_code = "RO"
start = pd.Timestamp("2026-01-01", tz="UTC")
end = pd.Timestamp("2026-01-08", tz="UTC")

if not country_code:
    raise ValueError("country_code must not be empty")
if not api_key:
    raise ValueError("api_key must not be empty")

start = start.tz_localize("UTC") if start.tzinfo is None else start.tz_convert("UTC")
end = end.tz_localize("UTC") if end.tzinfo is None else end.tz_convert("UTC")

if start >= end:
    raise ValueError(f"Invalid range: start ({start}) must be before end ({end})")

print("Country:", country_code)
print("Start:", start)
print("End:", end)

Country: RO
Start: 2026-01-01 00:00:00+00:00
End: 2026-01-08 00:00:00+00:00


In [3]:
truststore.inject_into_ssl()
client = EntsoePandasClient(api_key=api_key)

try:
    raw = client.query_load(country_code, start=start, end=end)
except NoMatchingDataError as exc:
    raise ValueError(
        f"ENTSO-E has no load data for {country_code} between {start} and {end}"
    ) from exc
except Exception as exc:
    raise RuntimeError("ENTSO-E API query failed") from exc

print("Shape:", raw.shape)
raw.head()

Shape: (672, 1)


,Actual Load
2026-01-01 02:00:00+02:00,5571.0
2026-01-01 02:15:00+02:00,5533.0
2026-01-01 02:30:00+02:00,5496.0
2026-01-01 02:45:00+02:00,5455.0
2026-01-01 03:00:00+02:00,5408.0


In [4]:
if isinstance(raw, pd.DataFrame):
    if raw.empty:
        raise ValueError("ENTSO-E API returned no data")
    if raw.shape[1] != 1:
        raise ValueError(f"Unexpected API response columns: {list(raw.columns)}")
    raw = raw.iloc[:, 0]

if raw.empty:
    raise ValueError("ENTSO-E API returned no data")

index = pd.DatetimeIndex(pd.to_datetime(raw.index, errors="raise"))
index = index.tz_localize("UTC") if index.tz is None else index.tz_convert("UTC")

series = pd.to_numeric(raw, errors="coerce")
series.index = index
series.index.name = "timestamp_utc"
series = series.sort_index().rename("load_mw")

if series.index.has_duplicates:
    raise ValueError("ENTSO-E API response contains duplicate timestamps")

series = series.resample("1h").mean().rename("load_mw")
series.index.name = "timestamp_utc"

if (series.dropna() < 0).any():
    raise ValueError("Negative load values present after resampling")

print("Shape:", series.shape)
print("Head:\n", series.head())

Shape: (168,)
Head:
 timestamp_utc
2026-01-01 00:00:00+00:00    5513.75
2026-01-01 01:00:00+00:00    5360.50
2026-01-01 02:00:00+00:00    5277.75
2026-01-01 03:00:00+00:00    5285.75
2026-01-01 04:00:00+00:00    5287.00
Freq: h, Name: load_mw, dtype: float64


In [5]:
print("Index tz:", series.index.tz)
print("Sorted:", series.index.is_monotonic_increasing)
print("Duplicates:", series.index.has_duplicates)
print("NaN count:", series.isna().sum())

expected_hours = int((end - start).total_seconds() // 3600)
print("Expected hours:", expected_hours)
print("Actual rows:", len(series))

Index tz: UTC
Sorted: True
Duplicates: False
NaN count: 0
Expected hours: 168
Actual rows: 168


In [6]:
out_path = Path("../data/raw/entsoe/load_RO_api_2026-01-01_2026-01-08.csv")
out_path.parent.mkdir(parents=True, exist_ok=True)
series.to_csv(out_path, index_label="timestamp_utc")
print("Saved to:", out_path)

Saved to: ..\data\raw\entsoe\load_RO_api_2026-01-01_2026-01-08.csv
